In [255]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel
import os
import umap
import plotly.express as px
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import ast
from pathlib import Path

In [164]:
TRAIN_CROP_FILES     = "/home/amitli/repo/dor6_vision/Dataset/train_crop_files/"
CSV_TRAIN_POINT_FILE = "/home/amitli/repo/dor6_vision/Dataset/train.csv"
CSV_TRAIN_CROP_EMBEDDING_FILE = "/home/amitli/repo/dor6_vision/Dataset/embeddings_train_crop.csv"

TEST_CROP_FILES_PATH = "/home/amitli/repo/dor6_vision/Dataset/test_set_crop/"
TEST_FULL_MODE_FILES_PATH = "/home/amitli/repo/dor6_vision/Dataset/test_set/"
CSV_TEST_POINT_FILE = "/home/amitli/repo/dor6_vision/Dataset/test_set_point.csv"
CSV_TEST_CROP_POINT_EMBEDDING = "/home/amitli/repo/dor6_vision/Dataset/train_test_crop_embeddings.csv"
CSV_TEST_CROP_CLASS_PREDICTION = "/home/amitli/repo/dor6_vision/Dataset/test_crop_class_prediction.csv"

CSV_TRAIN_TEST_CROP_UMAP = "/home/amitli/repo/dor6_vision/Dataset/train_test_crop_embeddings.csv"
CSV_TRAIN_TEST_CROP_HTML = "/home/amitli/repo/dor6_vision/Dataset/train_test_crop.html"

FONT_FILE = "/usr/share/fonts/truetype/freefont/FreeMono.ttf"

TEST_CROP_WITH_PREDICTION_PATH = "/home/amitli/repo/dor6_vision/results/test_set_crop_with_prediction/"
TEST_CROP_WITH_PREDICTION_HTML = "/home/amitli/repo/dor6_vision/results/test_crop_prediction.html"



In [10]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA: {device}")

CUDA: cuda


<h2> Load DINOv2 model: </h2>

In [29]:
def load_dino_model(model_id):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoImageProcessor.from_pretrained(model_id)
    model     = AutoModel.from_pretrained(model_id).to(device)
    return processor, model

processor, model = load_dino_model('facebook/dinov2-base')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


<h2> Calc train embeddings: </h2>

In [80]:
df_train = pd.read_csv(CSV_TRAIN_POINT_FILE)

l_jpg        = []
l_gt         = []
l_point_pred = []
l_point_x    = []
l_point_y    = []
l_emb        = []
l_file_type  = []

for i in tqdm(range(len(df_train))):

    jpg_file   = df_train["jpg_file"].values[i]
    jpg_file   = os.path.basename(jpg_file)
    gt         = df_train["gt"].values[i]
    point_pred = df_train["point_pred"].values[i]
    point_x    = df_train["x"].values[i]
    point_y    = df_train["y"].values[i]

    image_path = f"{TRAIN_CROP_FILES}{jpg_file}"
    image      = Image.open(image_path).convert('RGB')
    inputs     = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    embedding = embedding.reshape(-1)

    l_jpg        .append(jpg_file)
    l_gt         .append(gt)
    l_point_pred .append(point_pred)
    l_point_x    .append(point_x)
    l_point_y    .append(point_y)
    l_emb        .append(embedding)
    l_file_type  .append("Train")


100%|██████████| 15761/15761 [03:03<00:00, 85.87it/s]


In [83]:
df_train_emb = pd.DataFrame({"jpg_file"  : l_jpg,
                             "gt"        : l_gt,
                             "point_pred": l_point_pred,
                             "point_x"   : l_point_x,
                             "point_y"   : l_point_x,
                             "embedding" : l_emb,
                             "file_typ"  : l_file_type})

df_train_emb.to_csv(CSV_TRAIN_CROP_EMBEDDING_FILE, index=False)

<h2> Create test embeddings: </h2>

In [244]:
df_test = pd.read_csv(CSV_TEST_POINT_FILE)

l_jpg        = []
l_gt         = []
l_point_pred = []
l_point_x    = []
l_point_y    = []
l_emb        = []
l_file_type  = []

for i in tqdm(range(len(df_test))):

    jpg_file   = df_test["jpg_file"].values[i]
    jpg_file   = os.path.basename(jpg_file)
    gt         = "Test_Set"
    point_pred = df_test["point_pred"].values[i]
    point_x    = df_test["x"].values[i]
    point_y    = df_test["y"].values[i]

    image_path = f"{TEST_CROP_FILES_PATH}{jpg_file}"
    image      = Image.open(image_path).convert('RGB')
    inputs     = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    embedding = embedding.reshape(-1)

    l_jpg        .append(jpg_file)
    l_gt         .append(gt)
    l_point_pred .append(point_pred)
    l_point_x    .append(point_x)
    l_point_y    .append(point_y)
    l_emb        .append(embedding)
    l_file_type  .append("Test")


100%|██████████| 598/598 [00:07<00:00, 79.40it/s]


In [245]:
df_test_emb = pd.DataFrame({"jpg_file"  : l_jpg,
                             "gt"        : l_gt,
                             "point_pred": l_point_pred,
                             "point_x"   : l_point_x,
                             "point_y"   : l_point_y,
                             "embedding" : l_emb,
                             "file_typ"  : l_file_type})

df_test_emb.to_csv(CSV_TEST_CROP_POINT_EMBEDDING, index=False)
df_test_emb.head(2)

,jpg_file,gt,point_pred,point_x,point_y,embedding,file_typ
0,frame_7200.jpg,Test_Set,"<points coords=""1 1 320 329"">military vehicle<...",203.0,117.0,"[0.7831422, -0.0073285843, 0.69521207, 0.27724...",Test
1,frame_14100.jpg,Test_Set,"<points coords=""1 1 652 404"">military vehicle<...",414.0,143.0,"[-0.451442, -0.29531357, -0.69689155, -0.25843...",Test


<h2> Merga df_train, df_test and run UMAP </h2>

In [246]:
df_crop_train      = pd.read_csv(CSV_TRAIN_CROP_EMBEDDING_FILE)
df_crop_test       = pd.read_csv(CSV_TEST_CROP_POINT_EMBEDDING)
df_crop_train_test = pd.concat([df_crop_train, df_crop_test])
df_crop_train_test.shape

(16359, 7)

In [247]:
l_embbeddings   = df_crop_train_test[f'embedding'].apply(lambda x: np.fromstring(x.strip('[]'), sep=' '))
l_embbeddings   = np.vstack(l_embbeddings)
reducer         = umap.UMAP(n_components=2, random_state=42, metric='cosine')
X_umap          = reducer.fit_transform(l_embbeddings)


df_crop_train_test['umap_x']    = X_umap[:, 0]
df_crop_train_test['umap_y']    = X_umap[:, 1]

/home/amitli/repo/dor6_vision/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [248]:
df_crop_train_test.to_csv(CSV_TRAIN_TEST_CROP_UMAP, index=False)

<h2> Plot UMAP: </h2>

In [249]:
df_crop_train_test = pd.read_csv(CSV_TRAIN_TEST_CROP_UMAP)

In [250]:
df_crop_train_test['hover'] = df_crop_train_test["jpg_file"]
fig = px.scatter(df_crop_train_test, x='umap_x', y='umap_y', color='gt', hover_name="hover", title=f"#Samples: {len(df_crop_train_test)}")

fig.write_html(CSV_TRAIN_TEST_CROP_HTML)

In [93]:
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '<b>%{hovertext}</b><br><br>gt=SA-22<br>umap_x=%{x}<br>umap_y=%{y}<extra></extra>',
              'hovertext': array(['11-20-27_844400_489.jpg', '11-17-44_444400_591.jpg',
                                  '11-17-44_444400_172.jpg', ..., '11-20-27_844400_1227.jpg',
                                  '11-17-44_444400_1309.jpg', '11-20-27_844400_403.jpg'],
                                 shape=(5298,), dtype=object),
              'legendgroup': 'SA-22',
              'marker': {'color': '#636efa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'SA-22',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('djQO9buoLUDjiFo/okYVwNnIGppXeg' ... 'o2x7llL0Atsp3vp8YKwC4CY30D2ydA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('32qduBxPJkBBgXfy6akgQAd40sJlXS' ... 'palxoJJ0D27SQi/DMWQBCugEI91SVA'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': '<b>%{hovertext}</b><br><br>gt=SCUD<br>umap_x=%{x}<br>umap_y=%{y}<extra></extra>',
              'hovertext': array(['11-21-10_1324400_1213.jpg', '11-21-10_1324400_247.jpg',
                                  '11-21-10_1324400_353.jpg', ..., '11-21-10_1324400_824.jpg',
                                  '11-20-34_884400_1119.jpg', '11-20-34_884400_947.jpg'],
                                 shape=(5345,), dtype=object),
              'legendgroup': 'SCUD',
              'marker': {'color': '#EF553B', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'SCUD',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('8gaY+Q7uGEDPMotQbMUjQPd14JwRXS' ... '8N/xpALJ0PzxJEJ0BIiV3b220lQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('yJqRQe5+MkBqTl5kAjIwQGzrp/+sBT' ... 'iv4TFAW5TZIJNUE8B7Wcl9YSgRwA=='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': '<b>%{hovertext}</b><br><br>gt=T-90<br>umap_x=%{x}<br>umap_y=%{y}<extra></extra>',
              'hovertext': array(['11-20-40_924400_920.jpg', '11-20-40_924400_382.jpg',
                                  '11-21-17_1284400_961.jpg', ..., '11-21-17_1284400_899.jpg',
                                  '11-18-04_484400_821.jpg', '11-18-04_484400_620.jpg'],
                                 shape=(5118,), dtype=object),
              'legendgroup': 'T-90',
              'marker': {'color': '#00cc96', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'T-90',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('bHh6pSy7MECQSrGjcUAvQPr4Ol+iEB' ... 'Y/HlWwHED6YBkbuikEwKuP6uh2fsA/'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('919CYECiEEDTwVX5+YoHQPOrOUAwPx' ... 'No6J/gHUDKH2f1afDlPxKPej65PPQ/'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('<b>%{hovertext}</b><br><br>gt=' ... '<br>umap_y=%{y}<extra></extra>'),
              'hovertext': array(['frame_7200.jpg', 'frame_14100.jpg', 'frame_2910.jpg', ...,
                                  'frame_15930.jpg', 'frame_13590.jpg', 'frame_390.jpg'],
                                 shape=(598,), dtype=object),
              'legendgroup': 'Test_Set',
              'marker': {'color': '#ab63fa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Test_Set',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('cJo+O+DqBkC9UpYhjt0oQBypCIGmDx' ... 'FgCXofQCDvVSsTTiZATLUFOkkJDEA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('//3Omy+b1T9JN3gVCJH7v8lXAimxuy' ... 'Mn/r2wv275SEp6OCRAUz4EVaMvIEA='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'l

<h2> Prediction (embeddings): </h2>

In [269]:
df_train_set  = df_crop_train_test[df_crop_train_test.file_typ == "Train"]
df_test_set   = df_crop_train_test[df_crop_train_test.file_typ == "Test"]

l_labels      = df_train_set['gt'].values
l_embeddings  = df_train_set['embedding']
l_embeddings = l_embeddings.apply(lambda x: np.fromstring(x.strip('[]'), sep=' '))
l_embeddings = l_embeddings.values
l_embeddings = np.stack(l_embeddings)


l_jpg_file            = []
l_classification_pred = []
l_point_pred          = []
l_point_x             = []
l_point_y             = []
l_umap_x              = []
l_umap_y              = []

for i in tqdm(range(len(df_test_set))):

    new_emb  = df_test_set['embedding'].values[i]
    new_emb  = np.fromstring(new_emb.strip('[]'), sep=' ')
    new_emb = new_emb.reshape(1, -1)

    distances       = cdist(new_emb, l_embeddings, metric='cosine').flatten()
    K               = 10
    closest_indices = np.argsort(distances)[:K]
    neighbor_labels = l_labels[closest_indices]
    unique, counts  = np.unique(neighbor_labels, return_counts=True)
    relative_counts = counts / K
    class_scores    = dict(zip(unique, relative_counts))

    l_jpg_file            .append(df_test_set['jpg_file'].values[i])
    l_point_pred          .append(df_test_set['point_pred'].values[i])
    l_point_x             .append(df_test_set['point_x'].values[i])
    l_point_y             .append(df_test_set['point_y'].values[i])
    l_umap_x              .append(df_test_set['umap_x'].values[i])
    l_umap_y              .append(df_test_set['umap_y'].values[i])
    l_classification_pred .append(class_scores)

df_test_class_prediction = pd.DataFrame({"jpg_file"           : l_jpg_file,
                                         "point_pred"         : l_point_pred,
                                         "point_x"            : l_point_x,
                                         "point_y"            : l_point_y,
                                         "umap_x"             : l_umap_x,
                                         "umap_y"             : l_umap_y,
                                         "classification_pred": l_classification_pred})
df_test_class_prediction.to_csv(CSV_TEST_CROP_CLASS_PREDICTION, index=False)


100%|██████████| 598/598 [00:09<00:00, 65.97it/s]


<h2> Create jpg files with prediciton (crop files): </h2>

In [270]:
df_test_class_prediction = pd.read_csv(CSV_TEST_CROP_CLASS_PREDICTION)
df_test_class_prediction.head(3)

,jpg_file,point_pred,point_x,point_y,umap_x,umap_y,classification_pred
0,frame_7200.jpg,"<points coords=""1 1 320 329"">military vehicle<...",203.0,117.0,2.864686,0.337597,{'T-90': np.float64(1.0)}
1,frame_14100.jpg,"<points coords=""1 1 652 404"">military vehicle<...",414.0,143.0,12.432725,-1.722908,{'SCUD': np.float64(1.0)}
2,frame_2910.jpg,"<points coords=""1 1 346 433"">military vehicle<...",220.0,154.0,7.765284,14.366586,{'SCUD': np.float64(1.0)}


In [277]:
def get_text_nicely(image_path, text):

    jpg_file    = os.path.basename(image_path)
    if type(text) == str:
        text        = text.replace('np.float64', '')
        text        = ast.literal_eval(text)
    sorted_keys = sorted(text, key=lambda x: text[x], reverse=True)
    final       = f"{jpg_file}\n"
    for key in sorted_keys:
        final=final + f"{key} = {text[key]:.3f}\n"

    return final

def plot_img_with_point(image_path, x, y, text="", output_path=None):

    img        = Image.open(image_path).convert("RGB")
    draw       = ImageDraw.Draw(img)
    r          = 5  # radius of the dot
    left_up    = (x - r, y - r)
    right_down = (x + r, y + r)

    if os.path.exists(FONT_FILE):
        font = ImageFont.truetype(FONT_FILE, size=24)
    else:
        font = ImageFont.load_default()

    if np.isnan(x) == False:
        draw.ellipse([left_up, right_down], fill="red", outline="red")
        draw.text((3, 3), get_text_nicely(image_path, text), fill="red", font= font)
    #img.show()
    if output_path is not None:
        img.save(output_path)

In [278]:
for i in tqdm(range(len(df_test_class_prediction))):
    jpg_file            = df_test_class_prediction["jpg_file"].values[i]

    full_file_path      = f"{TEST_FULL_MODE_FILES_PATH}/{jpg_file}"
    point_pred          = df_test_class_prediction["point_pred"].values[i]
    point_x             = df_test_class_prediction["point_x"].values[i]
    point_y             = df_test_class_prediction["point_y"].values[i]
    classification_pred = df_test_class_prediction["classification_pred"].values[i]
    output_path         = f"{TEST_CROP_WITH_PREDICTION_PATH}/{jpg_file}"
    plot_img_with_point(full_file_path, point_x, point_y, classification_pred, output_path)


100%|██████████| 598/598 [00:02<00:00, 202.89it/s]


In [274]:
df_test_class_prediction[df_test_class_prediction.jpg_file == 'frame_10050.jpg']

,jpg_file,point_pred,point_x,point_y,umap_x,umap_y,classification_pred
199,frame_10050.jpg,"<points coords=""1 1 533 646"">military vehicle<...",338.0,229.0,-2.536484,7.907574,{'SA-22': np.float64(1.0)}


<h2> Create HTML Result: </h2>

In [275]:
def create_html_prediction():
    image_folder = TEST_CROP_WITH_PREDICTION_PATH
    output_html  = TEST_CROP_WITH_PREDICTION_HTML

    # Collect all jpg files
    image_paths = sorted(Path(image_folder).glob("*.jpg"))

    # Start HTML
    html = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Image Gallery</title>
        <style>
            body {
                font-family: Arial, sans-serif;
            }

            .grid {
                display: grid;
                grid-template-columns: repeat(1, 1fr); /* 1 columns */
                gap: 20px;
            }

            .item {
                text-align: center;
            }

            img {
                width: 100%;
                max-width: 250px;
                height: auto;
                display: block;
                margin: 0 auto;
            }

            .title {
                margin-top: 5px;
                font-size: 14px;
                word-break: break-all;
            }
        </style>
    </head>
    <body>

    <h2>Image Gallery</h2>

    <div class="grid">
    """

    # Add images
    for img_path in image_paths:
        filename = img_path.name
        title = os.path.splitext(filename)[0]

        html += f"""
        <div class="item">
            <img src="{image_folder}/{filename}" alt="{title}" loading="lazy">
            <div class="title">{title}</div>
        </div>
        """

    # Close HTML
    html += """
    </div>

    </body>
    </html>
    """

    # Save file
    with open(output_html, "w") as f:
        f.write(html)

    print(f"HTML gallery saved to {output_html}")

In [279]:
create_html_prediction()

HTML gallery saved to /home/amitli/repo/dor6_vision/results/test_crop_prediction.html


<h2> Debug </h2>

In [280]:
df_crop_train_test[df_crop_train_test.jpg_file == 'frame_6330.jpg']

,jpg_file,gt,point_pred,point_x,point_y,embedding,file_typ,umap_x,umap_y,hover
16337,frame_6330.jpg,Test_Set,"<points coords=""1 1 636 438"">military vehicle<...",404.0,155.0,[-3.42873544e-01 2.65593827e-01 4.45983201e-...,Test,2.796111,0.329056,frame_6330.jpg


In [263]:
df_train_set  = df_crop_train_test[df_crop_train_test.file_typ == "Train"]
l_labels      = df_train_set['gt'].values
l_embeddings  = df_train_set['embedding']
l_embeddings = l_embeddings.apply(lambda x: np.fromstring(x.strip('[]'), sep=' '))
l_embeddings = l_embeddings.values
l_embeddings = np.stack(l_embeddings)

l_embeddings.shape

(15761, 768)

In [266]:
df_tmp   = df_crop_train_test[df_crop_train_test.jpg_file == 'frame_10020.jpg']
new_emb  = df_tmp['embedding'].values[0]
new_emb  = np.fromstring(new_emb.strip('[]'), sep=' ')
new_emb = new_emb.reshape(1, -1)
new_emb.shape

(1, 768)

In [267]:
distances       = cdist(new_emb, l_embeddings, metric='cosine').flatten()
K               = 10
closest_indices = np.argsort(distances)[:K]
neighbor_labels = l_labels[closest_indices]
unique, counts  = np.unique(neighbor_labels, return_counts=True)
relative_counts = counts / K
class_scores    = dict(zip(unique, relative_counts))
class_scores

{'SA-22': np.float64(1.0)}